# SQLite-Datenbank: Strompreise und Gemeindedaten

**ZHAW Data Analytics HS2026 · Gruppenprojekt «Strompreise der Gemeinden» · Teil Metehan (Pflicht 3)**

Dieses Notebook baut aus den CSV-Dateien eine SQLite-Datenbank `strompreise.db` und fragt sie mit SQL aus Python ab.

**Eingabedateien** (im Ordner `daten/`):

| Datei | Inhalt | Quelle |
| --- | --- | --- |
| `Daten_Strompreise_2021.csv`, `_2024.csv`, `_2027.csv` | Alle Tarife pro Gemeinde, Netzbetreiber, Produkt und Kategorie | ElCom via LINDAS (SPARQL) |
| `regionalportraets_2021.csv` | Kennzahlen pro Gemeinde im Langformat (ein Wert pro Zeile) | BFS Regionalporträts 2021 |
| `strompreise_gemeinden.csv` | Lennys verbundene Tabelle, nur zur Kontrolle am Schluss | eigener Join |

**Datenmodell.** Statt alles in eine grosse Tabelle zu packen, wird jede Sache nur einmal gespeichert:

```
gemeinde (gemeinde_nr PK, name)
   │1
   │n
tarif (jahr, gemeinde_nr FK, netzbetreiber_id FK, produkt, kategorie,  ← PK aus diesen 5 Spalten
       total, energie, netznutzung, abgaben, netzzuschlag, messtarif, messtarif_chf_jahr)
   │n
   │1
netzbetreiber (netzbetreiber_id PK, name)

gemeinde 1 ── n  kennzahl (gemeinde_nr FK, indikator FK, wert, status)  n ── 1  indikator (code PK, bezeichnung, einheit, periode)
```

Darauf setzen zwei **Views** auf (gespeicherte Abfragen, keine Kopien der Daten):
- `v_preis_h4`: ein Preis pro Gemeinde und Jahr (Standardprodukt, Kategorie H4, Mittelwert über die Netzbetreiber)
- `v_analyse`: eine Zeile pro Gemeinde mit Preisen 2021/2024/2027 und Gemeindemerkmalen, die Tabelle für EDA und Modell

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

def finde_wurzel(start: Path) -> Path:
    """Klettert nach oben, bis daten/ und scripts/ nebeneinander liegen."""
    for ordner in [start, *start.parents]:
        if (ordner / "daten").is_dir() and (ordner / "scripts").is_dir():
            return ordner
    raise RuntimeError(f"Repo-Wurzel von {start} aus nicht gefunden")


WURZEL = finde_wurzel(Path.cwd().resolve())

DATEN = WURZEL / "daten"
DB = WURZEL / "sql" / "strompreise.db"
DB.parent.mkdir(parents=True, exist_ok=True)
JAHRE = [2021, 2024, 2027]

# Bei jedem Lauf neu aufbauen, damit das Notebook reproduzierbar ist
if DB.exists():
    DB.unlink()

con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON")   # SQLite prueft Fremdschluessel nur mit diesem Schalter
print("SQLite-Version:", sqlite3.sqlite_version)

SQLite-Version: 3.42.0


## 1 · Tabellen anlegen

`CREATE TABLE` legt die Struktur fest, bevor Daten hineinkommen. Wichtig sind drei Dinge:

- **Primärschlüssel (PK)** verhindern doppelte Zeilen. Ein Tarif ist eindeutig durch Jahr, Gemeinde, Netzbetreiber, Produkt und Kategorie.
- **Fremdschlüssel (FK)** stellen sicher, dass jede Gemeinde- und Netzbetreibernummer in `tarif` auch in ihrer Tabelle existiert.
- **CHECK** fängt unmögliche Werte ab, zum Beispiel ein falsches Produkt.

In [2]:
SCHEMA = '''
CREATE TABLE gemeinde (
    gemeinde_nr      INTEGER PRIMARY KEY,      -- BFS-Gemeindenummer
    name             TEXT                      -- vom BFS, sonst von der ElCom
);

CREATE TABLE netzbetreiber (
    netzbetreiber_id INTEGER PRIMARY KEY,      -- ElCom-Nummer des Netzbetreibers
    name             TEXT NOT NULL
);

CREATE TABLE tarif (
    jahr               INTEGER NOT NULL,
    gemeinde_nr        INTEGER NOT NULL REFERENCES gemeinde(gemeinde_nr),
    netzbetreiber_id   INTEGER NOT NULL REFERENCES netzbetreiber(netzbetreiber_id),
    produkt            TEXT    NOT NULL CHECK (produkt IN ('standard', 'cheapest')),
    kategorie          TEXT    NOT NULL,       -- H1-H8 Haushalte, C1-C7 Gewerbe
    total              REAL    NOT NULL,       -- alle Preise in Rp./kWh, exkl. MWST
    energie            REAL,
    netznutzung        REAL,
    abgaben            REAL,
    netzzuschlag       REAL,
    messtarif          REAL    NOT NULL DEFAULT 0,  -- erst ab 2026 separat, vorher 0
    messtarif_chf_jahr REAL,                   -- CHF pro Jahr, nur Info
    PRIMARY KEY (jahr, gemeinde_nr, netzbetreiber_id, produkt, kategorie)
);

CREATE TABLE indikator (
    code         TEXT PRIMARY KEY,             -- z.B. Ind_01_01
    bezeichnung  TEXT NOT NULL,
    einheit      TEXT,
    periode      TEXT,                         -- Bezugsjahr der Kennzahl beim BFS
    label_quelle TEXT                          -- 'geprueft' oder 'abgeleitet', siehe Abschnitt 3
);

CREATE TABLE kennzahl (
    gemeinde_nr INTEGER NOT NULL REFERENCES gemeinde(gemeinde_nr),
    indikator   TEXT    NOT NULL REFERENCES indikator(code),
    wert        REAL,                          -- leer, wenn das BFS keinen Wert ausweist
    status      TEXT,                          -- BFS-Status: A = Wert vorhanden, M/Q = kein Wert
    PRIMARY KEY (gemeinde_nr, indikator)
);

-- Index fuer die haeufigste Abfrage: ein Produkt und eine Kategorie ueber alle Gemeinden
CREATE INDEX idx_tarif_produkt_kat ON tarif (produkt, kategorie, jahr);
'''
con.executescript(SCHEMA)
pd.read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'index') ORDER BY type DESC, name", con)

,name,type
0,gemeinde,table
1,indikator,table
2,kennzahl,table
3,netzbetreiber,table
4,tarif,table
5,idx_tarif_produkt_kat,index
6,sqlite_autoindex_indikator_1,index
7,sqlite_autoindex_kennzahl_1,index
8,sqlite_autoindex_tarif_1,index


## 2 · Strompreise laden

Die drei Jahresdateien werden mit pandas eingelesen und zusammengehängt. Daraus entstehen drei Tabellen:

- `netzbetreiber`: jede Nummer einmal, mit dem Namen aus dem neusten Jahr
- `gemeinde`: jede Nummer einmal. Der Name kommt zuerst vom BFS, sonst aus dem neusten ElCom-Jahr. So bekommen auch die 120 Tarifzeilen von 2021 einen Namen, bei denen die ElCom keinen liefert (fusionierte Gemeinden). Keine Nummer geht verloren.
- `tarif`: alle Preiszeilen, ohne die Namen (die stehen in den anderen Tabellen)

Eingefügt wird mit `executemany` und `INSERT`, also mit echtem SQL. `pandas.to_sql` würde die Tabellen selbst anlegen und das Schema mit den Schlüsseln überschreiben.

In [3]:
preise = pd.concat(
    [pd.read_csv(DATEN / "strompreise" / f"Daten_Strompreise_{j}.csv") for j in JAHRE],
    ignore_index=True,
)
preise["messtarif"] = preise["messtarif"].fillna(0.0)
print(f"{len(preise):,} Preiszeilen eingelesen")
preise.groupby("jahr").size().rename("zeilen").to_frame()

143,837 Preiszeilen eingelesen


,zeilen
jahr,
2021,54210
2024,48187
2027,41440


In [4]:
# BFS-Daten jetzt schon einlesen, weil die Gemeindenamen von dort kommen
bfs = pd.read_csv(DATEN / "bfs" / "regionalportraets_2021.csv", sep=";", encoding="utf-8-sig", dtype=str)
bfs = bfs[bfs["CODE_REGION"] != "CH"].copy()          # Zeile fuer die ganze Schweiz weglassen
bfs["gemeinde_nr"] = bfs["CODE_REGION"].astype(int)
bfs["wert"] = pd.to_numeric(bfs["VALUE"], errors="coerce")

# Netzbetreiber: Name aus dem neusten Jahr
nb = (preise.sort_values("jahr")
            .drop_duplicates("netzbetreiber_id", keep="last")[["netzbetreiber_id", "netzbetreiber"]])

# Gemeinden: Name zuerst vom BFS, sonst aus dem neusten ElCom-Jahr
namen_bfs = bfs.drop_duplicates("gemeinde_nr").set_index("gemeinde_nr")["REGION"]
namen_elcom = (preise.dropna(subset=["gemeinde"]).sort_values("jahr")
                     .drop_duplicates("gemeinde_nr", keep="last").set_index("gemeinde_nr")["gemeinde"])
alle_nr = sorted(set(preise["gemeinde_nr"]) | set(bfs["gemeinde_nr"]))
gem = pd.DataFrame({"gemeinde_nr": alle_nr})
gem["name"] = gem["gemeinde_nr"].map(namen_bfs).fillna(gem["gemeinde_nr"].map(namen_elcom))

def zeilen(df):
    # pandas-Werte in Python-Typen umwandeln, fehlende Werte als None (= NULL in SQL)
    return [tuple(None if pd.isna(v) else (v.item() if hasattr(v, "item") else v) for v in row)
            for row in df.itertuples(index=False)]

con.executemany("INSERT INTO netzbetreiber VALUES (?, ?)", zeilen(nb))
con.executemany("INSERT INTO gemeinde VALUES (?, ?)", zeilen(gem))

tarif_spalten = ["jahr", "gemeinde_nr", "netzbetreiber_id", "produkt", "kategorie", "total", "energie",
                 "netznutzung", "abgaben", "netzzuschlag", "messtarif", "messtarif_chf_jahr"]
con.executemany(f"INSERT INTO tarif VALUES ({', '.join('?' * len(tarif_spalten))})",
                zeilen(preise[tarif_spalten]))
con.commit()

print(f"netzbetreiber: {len(nb):,} · gemeinde: {len(gem):,} "
      f"(davon ohne Namen: {gem['name'].isna().sum()}) · tarif: {len(preise):,}")

netzbetreiber: 609 · gemeinde: 2,225 (davon ohne Namen: 0) · tarif: 143,837


## 3 · BFS-Gemeindedaten laden

Die Regionalporträts kommen im **Langformat**: eine Zeile pro Gemeinde und Kennzahl. So bleiben sie auch in der Datenbank. Das ist flexibel, denn neue Kennzahlen brauchen keine neue Spalte. Die breite Form für die Analyse macht später eine View.

Die CSV enthält nur Codes (`Ind_01_01`). Die Bezeichnungen stehen in der Tabelle `indikator`.

**Achtung bei den Bezeichnungen:** Fünf Codes sind `geprueft`, weil ihre Werte genau mit Lennys Spalten übereinstimmen (Einwohner, Dichte, Fläche, Beschäftigte, Anteil 65+). Die übrigen sind `abgeleitet`: Sie wurden aus der Reihenfolge der BFS-Themen und aus Plausibilitätsprüfungen erschlossen, etwa Zürich mit 0.15 % Leerwohnungen und 32 % Ausländeranteil. Bevor ihr eine abgeleitete Kennzahl im Modell verwendet, vergleicht die Bezeichnung mit der Kopfzeile der BFS-Excel- oder ODS-Datei.

In [5]:
# (code, bezeichnung, einheit, label_quelle)
INDIKATOREN = [
    ("Ind_01_01", "Einwohner",                                   "Anzahl",           "geprueft"),
    ("Ind_01_02", "Veränderung der Einwohnerzahl",               "%",                "abgeleitet"),
    ("Ind_01_03", "Bevölkerungsdichte",                          "Einwohner pro km²", "geprueft"),
    ("Ind_01_04", "Anteil 0-19 Jahre",                           "%",                "abgeleitet"),
    ("Ind_01_05", "Anteil 20-64 Jahre",                          "%",                "abgeleitet"),
    ("Ind_01_06", "Anteil 65 Jahre und älter",                   "%",                "geprueft"),
    ("Ind_01_08", "Ausländeranteil",                             "%",                "abgeleitet"),
    ("Ind_01_09", "Rohe Heiratsziffer",                          "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_01_10", "Rohe Scheidungsziffer",                       "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_01_11", "Rohe Geburtenziffer",                         "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_01_12", "Rohe Sterbeziffer",                           "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_01_13", "Privathaushalte",                             "Anzahl",           "abgeleitet"),
    ("Ind_01_14", "Durchschnittliche Haushaltsgrösse",           "Personen",         "abgeleitet"),
    ("Ind_04_01", "Gesamtfläche",                                "km²",              "geprueft"),
    ("Ind_04_02", "Siedlungsfläche",                             "%",                "abgeleitet"),
    ("Ind_04_03", "Veränderung Siedlungsfläche",                 "ha",               "abgeleitet"),
    ("Ind_04_04", "Landwirtschaftsfläche",                       "%",                "abgeleitet"),
    ("Ind_04_05", "Veränderung Landwirtschaftsfläche",           "ha",               "abgeleitet"),
    ("Ind_04_06", "Wald und Gehölze",                            "%",                "abgeleitet"),
    ("Ind_04_07", "Unproduktive Fläche",                         "%",                "abgeleitet"),
    ("Ind_06_03", "Beschäftigte total",                          "Anzahl",           "geprueft"),
    ("Ind_06_04", "Beschäftigte 1. Sektor",                      "Anzahl",           "abgeleitet"),
    ("Ind_06_05", "Beschäftigte 2. Sektor",                      "Anzahl",           "abgeleitet"),
    ("Ind_06_06", "Beschäftigte 3. Sektor",                      "Anzahl",           "abgeleitet"),
    ("Ind_06_07", "Arbeitsstätten total",                        "Anzahl",           "abgeleitet"),
    ("Ind_06_08", "Arbeitsstätten 1. Sektor",                    "Anzahl",           "abgeleitet"),
    ("Ind_06_09", "Arbeitsstätten 2. Sektor",                    "Anzahl",           "abgeleitet"),
    ("Ind_06_10", "Arbeitsstätten 3. Sektor",                    "Anzahl",           "abgeleitet"),
    ("Ind_08_01", "Leerwohnungsziffer",                          "%",                "abgeleitet"),
    ("Ind_08_04", "Neu gebaute Wohnungen",                       "pro 1000 Einwohner", "abgeleitet"),
    ("Ind_11_01", "Sozialhilfequote",                            "%",                "abgeleitet"),
    ("Ind_14_01", "Wähleranteil FDP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_02", "Wähleranteil CVP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_03", "Wähleranteil SP (Nationalrat 2019)",          "%",                "abgeleitet"),
    ("Ind_14_04", "Wähleranteil SVP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_05", "Wähleranteil EVP/CSP (Nationalrat 2019)",     "%",                "abgeleitet"),
    ("Ind_14_06", "Wähleranteil GLP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_07", "Wähleranteil BDP (Nationalrat 2019)",         "%",                "abgeleitet"),
    ("Ind_14_08", "Wähleranteil PdA/Sol. (Nationalrat 2019)",    "%",                "abgeleitet"),
    ("Ind_14_09", "Wähleranteil Grüne (Nationalrat 2019)",       "%",                "abgeleitet"),
    ("Ind_14_10", "Wähleranteil kleine Rechtsparteien (Nationalrat 2019)", "%",      "abgeleitet"),
]
periode = bfs.groupby("INDICATORS")["PERIOD_REF"].first()
ind = pd.DataFrame(INDIKATOREN, columns=["code", "bezeichnung", "einheit", "label_quelle"])
ind["periode"] = ind["code"].map(periode)

fehlend = set(bfs["INDICATORS"]) - set(ind["code"])
assert not fehlend, f"Codes ohne Bezeichnung: {fehlend}"

con.executemany("INSERT INTO indikator (code, bezeichnung, einheit, label_quelle, periode) VALUES (?, ?, ?, ?, ?)",
                zeilen(ind[["code", "bezeichnung", "einheit", "label_quelle", "periode"]]))
con.executemany("INSERT INTO kennzahl VALUES (?, ?, ?, ?)",
                zeilen(bfs[["gemeinde_nr", "INDICATORS", "wert", "STATUS"]]))
con.commit()
print(f"indikator: {len(ind)} · kennzahl: {len(bfs):,} Zeilen für {bfs['gemeinde_nr'].nunique():,} Gemeinden")

indikator: 41 · kennzahl: 89,052 Zeilen für 2,172 Gemeinden


## 4 · Kontrollen

Bevor jemand mit der Datenbank rechnet, wird geprüft, ob beim Laden etwas verloren ging oder kaputt ist.

In [6]:
pd.read_sql('''
    SELECT 'gemeinde'      AS tabelle, COUNT(*) AS zeilen FROM gemeinde
    UNION ALL SELECT 'netzbetreiber', COUNT(*) FROM netzbetreiber
    UNION ALL SELECT 'tarif',         COUNT(*) FROM tarif
    UNION ALL SELECT 'indikator',     COUNT(*) FROM indikator
    UNION ALL SELECT 'kennzahl',      COUNT(*) FROM kennzahl
''', con)

,tabelle,zeilen
0,gemeinde,2225
1,netzbetreiber,609
2,tarif,143837
3,indikator,41
4,kennzahl,89052


In [7]:
# a) Fremdschluessel: liefert keine Zeilen, wenn alle Verweise stimmen
fk_fehler = con.execute("PRAGMA foreign_key_check").fetchall()
print("Verletzte Fremdschlüssel:", len(fk_fehler))

# b) Summenkontrolle direkt in SQL: Bestandteile muessen das Total ergeben
pd.read_sql('''
    SELECT jahr,
           COUNT(*) AS zeilen,
           SUM(ABS(total - (energie + netznutzung + abgaben + netzzuschlag + messtarif)) > 0.01) AS summe_falsch
    FROM tarif
    GROUP BY jahr
''', con)

Verletzte Fremdschlüssel: 0


,jahr,zeilen,summe_falsch
0,2021,54210,0
1,2024,48187,0
2,2027,41440,0


## 5 · Views für die Analyse

**`v_preis_h4`**: Gemäss Projektplan wird mit dem Standardprodukt und der Kategorie H4 gerechnet (4500 kWh pro Jahr). Hat eine Gemeinde mehrere Netzbetreiber, wird der **Mittelwert** genommen, wie in Lennys Join. `anzahl_netzbetreiber` zeigt, wo das passiert.

Ab 2026 steht der Messtarif separat, vorher steckte er in der Netznutzung. Damit die Jahre vergleichbar sind, gibt es die Spalte `netznutzung_inkl_mess`.

**`v_analyse`**: Eine Zeile pro Gemeinde, die in allen drei Jahren einen Preis hat und BFS-Daten besitzt. Die Kennzahlen werden mit `MAX(CASE WHEN ...)` vom Lang- ins Breitformat gedreht.

In [8]:
con.executescript('''
DROP VIEW IF EXISTS v_preis_h4;
CREATE VIEW v_preis_h4 AS
SELECT jahr,
       gemeinde_nr,
       COUNT(*)                     AS anzahl_netzbetreiber,
       AVG(total)                   AS total,
       AVG(energie)                 AS energie,
       AVG(netznutzung + messtarif) AS netznutzung_inkl_mess,
       AVG(abgaben)                 AS abgaben,
       AVG(netzzuschlag)            AS netzzuschlag
FROM tarif
WHERE produkt = 'standard' AND kategorie = 'H4'
GROUP BY jahr, gemeinde_nr;

DROP VIEW IF EXISTS v_analyse;
CREATE VIEW v_analyse AS
WITH merkmale AS (
    SELECT gemeinde_nr,
           MAX(CASE WHEN indikator = 'Ind_01_01' THEN wert END) AS einwohner,
           MAX(CASE WHEN indikator = 'Ind_01_03' THEN wert END) AS dichte,
           MAX(CASE WHEN indikator = 'Ind_04_01' THEN wert END) AS flaeche_total,
           MAX(CASE WHEN indikator = 'Ind_06_03' THEN wert END) AS beschaeftigte,
           MAX(CASE WHEN indikator = 'Ind_01_06' THEN wert END) AS anteil_65plus
    FROM kennzahl
    GROUP BY gemeinde_nr
)
SELECT g.gemeinde_nr,
       g.name                 AS gemeinde,
       p21.anzahl_netzbetreiber,
       p21.total              AS total_2021,
       p24.total              AS total_2024,
       p27.total              AS total_2027,
       m.einwohner, m.dichte, m.flaeche_total, m.beschaeftigte, m.anteil_65plus
FROM gemeinde g
JOIN v_preis_h4 p21 ON p21.gemeinde_nr = g.gemeinde_nr AND p21.jahr = 2021
JOIN v_preis_h4 p24 ON p24.gemeinde_nr = g.gemeinde_nr AND p24.jahr = 2024
JOIN v_preis_h4 p27 ON p27.gemeinde_nr = g.gemeinde_nr AND p27.jahr = 2027
JOIN merkmale  m   ON m.gemeinde_nr   = g.gemeinde_nr;
''')
analyse = pd.read_sql("SELECT * FROM v_analyse", con)
print(f"v_analyse: {len(analyse):,} Gemeinden")
analyse.head()

v_analyse: 2,087 Gemeinden


,gemeinde_nr,gemeinde,anzahl_netzbetreiber,total_2021,total_2024,total_2027,einwohner,dichte,flaeche_total,beschaeftigte,anteil_65plus
0,1,Aeugst am Albis,1,17.122,31.609,23.872,1981.0,250.442478,7.91,442.0,18.021201
1,2,Affoltern am Albis,1,17.122,31.609,23.872,12303.0,1161.756374,10.59,6920.0,18.166301
2,3,Bonstetten,1,17.122,31.609,23.872,5572.0,749.932705,7.43,1014.0,16.529074
3,4,Hausen am Albis,1,17.122,31.609,23.872,3751.0,275.808824,13.60,1021.0,17.995201
4,5,Hedingen,1,17.122,31.609,23.872,3778.0,578.560490,6.53,1478.0,16.728428


## 6 · SQL-Abfragen aus Python

Die Abfragen laufen mit `pd.read_sql(sql, con)` und kommen direkt als DataFrame zurück. Gezeigt werden die Bausteine, die das Projekt braucht: Filtern, Gruppieren, Joins und Fallunterscheidungen.

**6.1 · Preisentwicklung H4:** Durchschnitt, Minimum und Maximum pro Jahr. SQLite hat keine Median-Funktion, darum wird der Median in pandas ergänzt.

In [9]:
entwicklung = pd.read_sql('''
    SELECT jahr,
           COUNT(*)            AS gemeinden,
           ROUND(AVG(total),2) AS mittelwert,
           ROUND(MIN(total),2) AS minimum,
           ROUND(MAX(total),2) AS maximum
    FROM v_preis_h4
    GROUP BY jahr
    ORDER BY jahr
''', con)
median = pd.read_sql("SELECT jahr, total FROM v_preis_h4", con).groupby("jahr")["total"].median().round(2)
entwicklung["median"] = entwicklung["jahr"].map(median)
entwicklung

,jahr,gemeinden,mittelwert,minimum,maximum,median
0,2021,2201,20.86,7.96,27.65,20.63
1,2024,2140,32.00,10.22,50.62,32.15
2,2027,2109,27.24,10.56,45.88,26.91


**6.2 · Welcher Preisbestandteil hat sich bewegt?** Durchschnitt der Bestandteile pro Jahr in Rp./kWh. Das ist die Grundlage für Unterfrage 1.

In [10]:
pd.read_sql('''
    SELECT jahr,
           ROUND(AVG(energie),2)               AS energie,
           ROUND(AVG(netznutzung_inkl_mess),2) AS netznutzung_inkl_mess,
           ROUND(AVG(abgaben),2)               AS abgaben,
           ROUND(AVG(netzzuschlag),2)          AS netzzuschlag,
           ROUND(AVG(total),2)                 AS total
    FROM v_preis_h4
    GROUP BY jahr
    ORDER BY jahr
''', con)

,jahr,energie,netznutzung_inkl_mess,abgaben,netzzuschlag,total
0,2021,7.95,9.76,0.85,2.29,20.86
1,2024,15.59,13.22,0.88,2.30,32.00
2,2027,11.19,12.86,0.90,2.30,27.24


**6.3 · Die zehn teuersten Gemeinden 2027** mit Join auf `gemeinde` und `netzbetreiber`.

In [11]:
pd.read_sql('''
    SELECT g.name AS gemeinde, n.name AS netzbetreiber, t.total
    FROM tarif t
    JOIN gemeinde      g ON g.gemeinde_nr      = t.gemeinde_nr
    JOIN netzbetreiber n ON n.netzbetreiber_id = t.netzbetreiber_id
    WHERE t.jahr = 2027 AND t.produkt = 'standard' AND t.kategorie = 'H4'
    ORDER BY t.total DESC
    LIMIT 10
''', con)

,gemeinde,netzbetreiber,total
0,Kestenholz,Energie Kestenholz,45.883
1,Niederhelfenschwil,"Politische Gemeinde Niederhelfenschwil, Elektr...",45.817
2,Eggersriet,Elektra- und Wasserkorporation Grub SG,41.683
3,Hauptwil-Gottshaus,Technische Gemeindebetriebe Bischofszell,41.048
4,Niederbüren,Technische Gemeindebetriebe Bischofszell,41.048
5,Hohentannen,Technische Gemeindebetriebe Bischofszell,41.048
6,Bischofszell,Technische Gemeindebetriebe Bischofszell,41.048
7,Zihlschlacht-Sitterdorf,Technische Gemeindebetriebe Bischofszell,41.048
8,Sils im Engadin/Segl,Azienda Elettrica Grevasalvas,40.930
9,Braunau,"Politische Gemeinde Braunau, Elektrizitätsvers...",38.883


**6.4 · Preis nach Gemeindegrösse:** Gemeinden werden mit `CASE` in Grössenklassen eingeteilt. Das ist eine erste Antwort auf Unterfrage 2.

In [12]:
pd.read_sql('''
    SELECT CASE
             WHEN einwohner <   1000 THEN '1: unter 1000'
             WHEN einwohner <   5000 THEN '2: 1000-4999'
             WHEN einwohner <  20000 THEN '3: 5000-19999'
             ELSE                         '4: ab 20000'
           END                                    AS groessenklasse,
           COUNT(*)                               AS gemeinden,
           ROUND(AVG(total_2021),2)               AS preis_2021,
           ROUND(AVG(total_2024),2)               AS preis_2024,
           ROUND(AVG(total_2027),2)               AS preis_2027,
           ROUND(AVG(total_2024 - total_2021),2)  AS anstieg_21_24
    FROM v_analyse
    GROUP BY groessenklasse
    ORDER BY groessenklasse
''', con)

,groessenklasse,gemeinden,preis_2021,preis_2024,preis_2027,anstieg_21_24
0,1: unter 1000,683,21.41,31.55,27.12,10.14
1,2: 1000-4999,1011,20.71,32.04,27.40,11.33
2,3: 5000-19999,343,20.28,32.61,27.14,12.33
3,4: ab 20000,50,20.44,32.57,26.84,12.13


**6.5 · Grösse des Netzbetreibers:** Wie viele Gemeinden versorgt ein Betreiber, und hängt das mit dem Preis zusammen? Hier wird eine Unterabfrage mit einem Join kombiniert. Auch das ist ein Merkmal für Unterfrage 2.

In [13]:
pd.read_sql('''
    WITH groesse AS (
        SELECT netzbetreiber_id, COUNT(DISTINCT gemeinde_nr) AS anzahl_gemeinden
        FROM tarif
        WHERE jahr = 2027 AND produkt = 'standard' AND kategorie = 'H4'
        GROUP BY netzbetreiber_id
    )
    SELECT CASE
             WHEN gr.anzahl_gemeinden = 1   THEN '1: eine Gemeinde'
             WHEN gr.anzahl_gemeinden <= 10 THEN '2: 2-10 Gemeinden'
             ELSE                                '3: mehr als 10'
           END                       AS betreibergroesse,
           COUNT(DISTINCT t.netzbetreiber_id) AS betreiber,
           COUNT(*)                  AS tarife,
           ROUND(AVG(t.total),2)     AS preis_2027
    FROM tarif t
    JOIN groesse gr ON gr.netzbetreiber_id = t.netzbetreiber_id
    WHERE t.jahr = 2027 AND t.produkt = 'standard' AND t.kategorie = 'H4'
    GROUP BY betreibergroesse
    ORDER BY betreibergroesse
''', con)

,betreibergroesse,betreiber,tarife,preis_2027
0,1: eine Gemeinde,421,421,28.57
1,2: 2-10 Gemeinden,90,313,27.65
2,3: mehr als 10,25,1508,26.89


## 7 · Kontrolle gegen Lennys Join

Die View `v_analyse` muss dieselben Gemeinden und dieselben Preise liefern wie Lennys Datei `strompreise_gemeinden.csv`. Wenn das stimmt, rechnen Datenbank und Join mit denselben Zahlen.

In [14]:
lenny = pd.read_csv(DATEN / "verbunden" / "strompreise_gemeinden.csv")
vergleich = analyse.merge(lenny, on="gemeinde_nr", how="outer", suffixes=("_db", "_lenny"), indicator=True)

print("Gemeinden nur in der DB:   ", (vergleich["_merge"] == "left_only").sum())
print("Gemeinden nur bei Lenny:   ", (vergleich["_merge"] == "right_only").sum())
beide = vergleich[vergleich["_merge"] == "both"]
for spalte in ["total_2021", "total_2024", "total_2027", "einwohner", "dichte",
               "flaeche_total", "beschaeftigte", "anteil_65plus"]:
    a, b = beide[f"{spalte}_db"], beide[f"{spalte}_lenny"]
    abweichung = ((a - b).abs() > 1e-6) | (a.isna() != b.isna())
    print(f"{spalte:<15} abweichende Werte: {abweichung.sum()}")

Gemeinden nur in der DB:    0
Gemeinden nur bei Lenny:    0
total_2021      abweichende Werte: 0
total_2024      abweichende Werte: 0
total_2027      abweichende Werte: 0
einwohner       abweichende Werte: 0
dichte          abweichende Werte: 0
flaeche_total   abweichende Werte: 0
beschaeftigte   abweichende Werte: 0
anteil_65plus   abweichende Werte: 0


## 8 · So arbeitet die Gruppe mit der Datenbank weiter

Die Datei `strompreise.db` ist alles, was Subraj und Lenny brauchen. Ein Lauf dieses Notebooks erzeugt sie neu.

```python
import sqlite3, pandas as pd
con = sqlite3.connect("strompreise.db")
df  = pd.read_sql("SELECT * FROM v_analyse", con)   # fertige Analysetabelle
```

- **Subraj (EDA, Modell):** startet mit `v_analyse`. Weitere Merkmale kommen als zusätzliche `MAX(CASE WHEN ...)`-Zeile in die View.
- **Lenny (Karte):** verbindet `gemeinde_nr` mit den Gemeindegrenzen.
- **PostgreSQL (Bonus 2):** Das Schema aus Abschnitt 1 läuft fast unverändert. Aus `REAL` wird `NUMERIC` oder `DOUBLE PRECISION`, und `PRAGMA` fällt weg.

In [15]:
con.close()
print(f"Fertig: {DB} ({DB.stat().st_size / 1e6:.1f} MB)")

Fertig: /Users/lennyfrei/Desktop/data-prices-electronic/sql/strompreise.db (24.1 MB)
